# Train the Cited Newsroom Verifier SLM (Qwen3-1.7B + QLoRA)

This notebook runs the **training leg** of your project on a free Google Colab GPU.
Your Mac handles data-gen and eval; this notebook handles fine-tuning.

## How to run it (you don't need to know how to train — just follow these)

1. **Open this notebook in Colab.** (Upload `train.ipynb` at https://colab.research.google.com
   → File → Upload notebook, or open it from your GitHub repo.)
2. **Turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
3. **Run everything:** menu **Runtime → Run all**. That's it — the notebook pulls your
   data from GitHub automatically, so there's nothing to upload.
4. Wait ~40 min. When it finishes, your browser will **download 4 files**
   (the trained adapter + two prediction files + `results.md`). Keep them.

**If a cell ever errors:** re-run that one cell (Colab sometimes needs a second pass
right after installing Unsloth — if it asks you to "Restart runtime," click Restart,
then Runtime → Run all again).

**What each stage does:** GPU check → install → pull files from GitHub → load base
model → **baseline predictions (before training)** → attach LoRA + **train** →
**tuned predictions (after training)** → score base-vs-tuned → download everything.

## Step 1 - Confirm you have a GPU

If the next cell errors with "command not found" or shows no GPU, you forgot to set
**Runtime -> Change runtime type -> T4 GPU**. Fix that, then re-run.

In [ ]:
!nvidia-smi

## Step 2 - Install Unsloth

This takes ~2-3 minutes. If you hit a weird dependency error, re-run this cell once
(Colab sometimes needs a second pass). It may ask you to restart the runtime the
first time - if so, click **Restart**, then continue from the next cell.

In [ ]:
%%capture
# Unsloth = the fast, low-memory QLoRA trainer. This is the whole reason a 1.7B
# model fits on a free T4 GPU. Unsloth pulls in matching transformers/trl/peft.
!pip install unsloth
# If you ever get a version-mismatch error, uncomment the force-reinstall below:
# !pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

## Step 3 - Get your files (automatic — no uploading)

This pulls `eval.py` and your datasets straight from your GitHub repo, so you don't
have to upload anything by hand. Just run the cell. It clones the repo and copies the
three files you need into Colab's working folder.

*(If you ever want to use different files, edit `SFT_FILE` / `TESTSET_FILE` in the
next cell.)*

In [ ]:
# Pull your project files from GitHub (no manual upload needed).
import os, shutil

REPO   = "https://github.com/oshikanoma/slm.git"
BRANCH = "dataset-and-training"   # branch these files live on

if os.path.isdir("slm_repo"):
    shutil.rmtree("slm_repo")
!git clone --depth 1 --branch {BRANCH} {REPO} slm_repo

# Copy the three files Colab needs into the working dir (flat, by name).
for src in ["slm_repo/eval.py",
            "slm_repo/data/train.sft.jsonl",
            "slm_repo/data/golden.json"]:
    shutil.copy(src, os.path.basename(src))

print("\nReady:", [f for f in ("eval.py", "train.sft.jsonl", "golden.json") if os.path.exists(f)])

In [ ]:
# These match the files pulled from GitHub in the previous cell. No need to change them.
SFT_FILE     = "train.sft.jsonl"  # your 1,000 training examples (chat format)
TESTSET_FILE = "golden.json"      # your 115-record eval golden set (base-vs-tuned)

import os
for f in (SFT_FILE, TESTSET_FILE, "eval.py"):
    assert os.path.exists(f), f"Missing {f} - re-run the previous cell (Step 3)."
print("All required files present.")

## Step 4 - Load the base model

`unsloth/Qwen3-1.7B` in 4-bit. This is the exact model you'll fine-tune, loaded in a
memory-efficient form so it fits the free GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

# 3072 covers 100% of your training records (longest ~2.2k tokens); 2048 would
# silently TRUNCATE ~2% of them and corrupt their JSON training target.
MAX_SEQ_LEN = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B",
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,     # QLoRA: 4-bit frozen base
    dtype = None,            # auto (bf16/fp16)
)
print("Base model loaded.")

## Step 5 - Baseline predictions (BEFORE training)

Critical: we capture how the **untrained** base model does on your test set *first*.
This is the "base" column of your base-vs-tuned table - the thing your fine-tune has
to beat. We reuse `build_messages` from your `eval.py` so the prompt format is
identical to training and to scoring. `enable_thinking=False` keeps the output as
clean JSON (no reasoning traces).

In [ ]:
import json
from eval import load_testset, build_messages

scenarios = load_testset(TESTSET_FILE)
print(f"Loaded {len(scenarios)} test scenarios.")

def generate_predictions(model, tokenizer, scenarios, out_path, max_new_tokens=512):
    """Run the model over each scenario, write eval-compatible predictions JSONL."""
    FastLanguageModel.for_inference(model)  # 2x faster inference mode
    with open(out_path, "w") as f:
        for i, scn in enumerate(scenarios, 1):
            messages = build_messages(scn)  # [system, user] from eval.py
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False,  # Qwen3: no reasoning trace -> clean JSON
            )
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
            f.write(json.dumps({"id": scn.id, "output": text}, ensure_ascii=False) + "\n")
            if i % 10 == 0 or i == len(scenarios):
                print(f"  {i}/{len(scenarios)}")
    print(f"Wrote {out_path}")

generate_predictions(model, tokenizer, scenarios, "base_preds.jsonl")

## Step 6 - Add LoRA adapters

We don't retrain the whole model - we attach small trainable "adapter" matrices
(LoRA) and train only those. `r` and `lora_alpha` are the main knobs; the defaults
below are the standard starting point. Remember the project rule: **fix problems in
your data, not by fiddling these numbers.**

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                       # LoRA rank
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing = "unsloth",  # saves memory
    random_state = 3407,
)
print("LoRA adapters attached. Trainable params:")
model.print_trainable_parameters()

## Step 7 - Prepare the training data

Your `*.sft.jsonl` has one record per line: `{"id", "messages":[system, user, assistant]}`.
We turn each into a single training string using Qwen3's chat template (thinking off),
so the model learns: given (system + user), produce the assistant's JSON verdicts.

In [ ]:
from datasets import Dataset

rows = [json.loads(l) for l in open(SFT_FILE)]
print(f"Loaded {len(rows)} training examples.")

def to_text(rec):
    return {
        "text": tokenizer.apply_chat_template(
            rec["messages"], tokenize=False, enable_thinking=False,
        )
    }

train_ds = Dataset.from_list(rows).map(to_text, remove_columns=["messages", "id"])
print("\nExample formatted training text:\n")
print(train_ds[0]["text"][:1200])

## Step 8 - Train

This is the main event. `train_on_responses_only` makes the model learn only from
the assistant's JSON output (it doesn't waste capacity memorizing your prompts).

On a free T4 this runs **3 epochs over your 1,000 examples (~375 steps) in roughly
30–45 minutes.** You don't need to do anything — just let it run.

Watch the **loss** number in the output drop as it goes — that's the model learning.
A curve falling from ~1.5–2.0 toward ~0.5–1.0 is healthy. (It won't hit zero, and it
shouldn't — that would mean memorizing.)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size = 8
        warmup_steps = 5,
        num_train_epochs = 3,              # real run: full 3 epochs over 1,000 examples
        # max_steps = 60,                  # (smoke-test cap — leave commented for the real run)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Train only on the assistant's response (Qwen3 chat markers).
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()
print("Done training.")

## Step 9 - Tuned predictions (AFTER training)

Same test set, same prompts - now with the trained adapter. This is the "tuned"
column of your table.

In [ ]:
generate_predictions(model, tokenizer, scenarios, "tuned_preds.jsonl")

## Step 10 - Quick score right here (optional)

`eval.py`'s deterministic metrics are pure Python, so you can see base-vs-tuned
numbers immediately in Colab. (The LLM-as-judge part needs your API key and is
better run on your Mac.) This is your **midweek gate**: watch `knowledge_leakage_rate`
and `fabricated_citation_rate` drop from base to tuned.

In [ ]:
# In Colab, {TESTSET_FILE} injects the Python variable into the shell command.
!python eval.py score --testset {TESTSET_FILE} --base base_preds.jsonl --tuned tuned_preds.jsonl --out results.md

## Step 11 - Save the adapter and download everything

Saves the trained LoRA adapter (tiny - a few MB), zips it, and downloads it along
with `base_preds.jsonl`, `tuned_preds.jsonl`, and `results.md`.

**Your browser may ask permission to download multiple files — click Allow.** If a
download doesn't start, the files are still in Colab's file browser (folder icon on
the left) — you can download them from there by right-clicking.

Back on your Mac you can re-run the full eval **including the LLM-as-judge** (the
calibration step from your notes):
```bash
python3 eval.py score --testset data/golden.json \
    --base base_preds.jsonl --tuned tuned_preds.jsonl --judge --out results.md
```

**Colab wipes everything when the session ends - so actually download the files.**

In [ ]:
ADAPTER_DIR = "qwen3-verifier-lora"
model.save_pretrained(ADAPTER_DIR)      # LoRA adapter only (small)
tokenizer.save_pretrained(ADAPTER_DIR)
!zip -r -q {ADAPTER_DIR}.zip {ADAPTER_DIR}
print("Saved + zipped adapter.")

from google.colab import files
for fname in [f"{ADAPTER_DIR}.zip", "base_preds.jsonl", "tuned_preds.jsonl", "results.md"]:
    try:
        files.download(fname)
    except Exception as e:
        print(f"(skip {fname}: {e})")

## Appendix (later in the week) - Publish to Hugging Face Hub

Your final submission needs the model on the HF Hub. When you're ready, get a token
from huggingface.co/settings/tokens (write access), then run the cell below. Skip
this for now - it's a Day 5 task, not needed to train.

In [ ]:
# from huggingface_hub import login
# login()  # paste your write token when prompted
# HF_REPO = "your-username/qwen3-1.7b-newsroom-verifier"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print("Pushed to", HF_REPO)